In [19]:
import os
from pathlib import Path
from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession
import pandas as pd

# 1. Move to project root if we are in the notebooks folder
if Path.cwd().name == "notebooks":
    os.chdir("..")

# 2. Initialize Kedro
project_path = Path.cwd()
bootstrap_project(project_path)

# 3. Create session and get catalog
session = KedroSession.create(project_path)
context = session.load_context()
catalog = context.catalog

print(f"✅ Kedro context loaded! Project root: {project_path}")

[04/21/26 22:25:10] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=664231;file://c:\anaconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py\plugin.py]8;;\:]8;id=165473;file://c:\anaconda3\envs\unis\Lib\site-packages\kedro_telemetry\plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

✅ Kedro context loaded! Project root: g:\Unidades compartidas\Alianzas\3. Data\UNIS\unis-perm-flow


In [20]:
# 2. Importar todas las funciones del archivo nodes.py
import unis_perm_flow.pipelines.data_processing.nodes as nodes

# Esto te dará una lista limpia de todo lo definido en ese archivo de nodos
print(dir(nodes))

['Tuple', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '__warningregistry__', 'check_and_export_duplicates', 'check_dataframe', 'clean_column_names', 'clean_column_objects', 'clean_nulls', 'convert_all_standardized_dates', 'convert_standardized_dates', 'datetime', 'duckdb', 'np', 'numeric_conversion_node', 'pd', 're', 'remove_accents', 'remove_accents_and_special_chars', 'select_columns', 'unicodedata', 'unis_preprocessing_calaca', 'unis_preprocessing_estaca']


In [21]:
parameters = catalog.load('parameters')

[04/21/26 22:25:16] INFO     Loading data from parameters (MemoryDataset)...                   ]8;id=840244;file://c:\anaconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=812786;file://c:\anaconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

In [22]:
# Base de unis con fechas de bajas, fechas de graduación
unis_caracterizacion = catalog.load('unis_caracterizacion')

[04/21/26 22:25:18] INFO     Loading data from unis_caracterizacion (CSVDataset)...            ]8;id=764931;file://c:\anaconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=529373;file://c:\anaconda3\envs\unis\Lib\site-packages\kedro\io\data_catalog.py#1048\1048]8;;\

## Parametros

In [13]:
unis_col_fechas = ['fecha_creacion', 'fecha_nacimiento', 'inicio_clases']
unis_col_emails = ['correo']
unis_col_dd = ['identidad']
unis_col_sort = ['identidad']

In [11]:
unis_caracterizacion = nodes.clean_column_names(unis_caracterizacion)

In [16]:
# Limpieza de la base de operaciones
unis_caracterizacion = nodes.clean_column_names(unis_caracterizacion)
unis_caracterizacion = nodes.convert_all_standardized_dates(unis_caracterizacion,unis_col_fechas)
unis_caracterizacion['identidad'] = pd.to_numeric(unis_caracterizacion['identidad'], errors='coerce')
unis_caracterizacion = nodes.clean_column_objects(unis_caracterizacion,unis_col_emails)
unis_caracterizacion_sd, unis_caracterizacion_cd  = nodes.check_and_export_duplicates(unis_caracterizacion, subset=unis_col_dd, col_ordenar = unis_col_sort)
unis_caracterizacion = unis_caracterizacion.drop(columns='nivel') 


In [18]:
unis_caracterizacion.columns


Index(['fecha_creacion', 'vendedor_inicial', 'id_crm', 'id_siu', 'matricula',
       'estado_inscripcion', 'id_alumno', 'tipo_identidad', 'identidad',
       'primer_nombre', 'segundo_nombre', 'apellido_paterno',
       'apellido_materno', 'genero', 'fecha_nacimiento', 'correo',
       'telefono_movil', 'telefono_casa', 'nacionalidad', 'tipo_alumno',
       'codigo_campus', 'campus', 'codigo_programa', 'programa', 'periodo',
       'inicio_clases', 'tipo_admision', 'apellido_de_casada',
       'telefono_trabajo', 'pais', 'departamento', 'municipio',
       'codigo_postal', 'zona', 'direccion_1', 'direccion_2',
       'area_profesional', 'universidad_de_origen',
       '_requiere_recibo_a_nombre_de_terceros_', 'nit_del_titular_del_recibo',
       'nombre_completo_del_titular_del_recibo',
       'direccion_del_titular_del_recibo',
       'direccion_adicional_del_titular_del_recibo'],
      dtype='str')

## Calendario Académico

In [ ]:
# Calendario Académico
unis_estados_calac = catalog.load("unis_estados_calac")

### Parámetros

In [ ]:
unis_caracterizacion = unis_estados_calac.merge(unis_caracterizacion, how='left', left_on=['identificacion'], right_on=['identidad'])

In [17]:
import pandas as pd

def transformar_caracterizacion_unis(
    df_caracterizacion: pd.DataFrame,
    df_estados_calac: pd.DataFrame,
    params_fechas: list,
    params_emails: list,
    params_duplicados: list,
    params_orden: list,
    params_columnas_caracterizacion: list
) -> pd.DataFrame:
    """
    Función para limpiar la base de operaciones y cruzarla con el calendario académico.
    """
    
    # 1. Limpieza de nombres de columnas
    df = nodes.clean_column_names(df_caracterizacion)
    
    # 2. Conversión de fechas usando los parámetros del YML
    df = nodes.convert_all_standardized_dates(df, params_fechas)
    
    # 3. Conversión de identidad a numérico (Coerce para manejar errores)
    df['identidad'] = pd.to_numeric(df['identidad'], errors='coerce')
    
    # 4. Limpieza de correos/objetos
    df = nodes.clean_column_objects(df, params_emails)
    
    # 5. Manejo de duplicados (obtenemos la base sin duplicados 'sd')
    df_sd, _ = nodes.check_and_export_duplicates(
        df, 
        subset=params_duplicados, 
        col_ordenar=params_orden
    )
    
    # 6. Eliminar columna 'nivel' si existe
    df_sd = df_sd.drop(columns='nivel', errors='ignore')

    # 7. Seleccionar columnas especificas
    df_sd = nodes.select_columns(df_sd, params_columnas_caracterizacion)

    
    # 8. Cruce con Calendario Académico (Merge)
    # Nota: Usamos df_sd para asegurar que el cruce no duplique filas inesperadamente
    resultado = df_estados_calac.merge(
        df_sd, 
        how='left', 
        left_on=['identificacion'], 
        right_on=['identidad']
    )
    
    return resultado

In [ ]:
df_estados_calac_caracterizacion = transformar_caracterizacion_central(
    df_caracterizacion=df_caracterizacion,
    df_estados_calac=df_estados_calac,
    params_fechas=parameters['central_caracterizacion_params']['col_fechas'],
    params_emails=parameters['central_caracterizacion_params']['col_emails'],
    params_duplicados=parameters['central_caracterizacion_params']['col_dd'],
    params_orden=parameters['central_caracterizacion_params']['col_sort'],
    params_columnas_caracterizacion=parameters['central_caracterizacion_params']['params_columnas_caracterizacion']
)